In [1]:
#import and setup

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import random_split,TensorDataset,DataLoader,Subset
from sklearn.metrics import precision_score, recall_score, f1_score,confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Single Machine Inference

model = nn.Sequential(
    nn.Linear(8,16),
    nn.ReLU(),
    nn.Linear(16,8),
    nn.ReLU(),
    nn.Linear(8,2)
).to(device)
checkpoint = torch.load(
    "../model/best_model.pth",
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])

mean = checkpoint["mean"]
std = checkpoint["std"]
threshold = checkpoint["threshold"]
feature_columns = checkpoint["feature_columns"]



new_data = {"Air temperature [K]" : 298.2,
"Process temperature [K]" : 309.4,
"Rotational speed [rpm]"  : 1439,
"Torque [Nm]"             : 37.3,
"Tool wear [min]"         : 58,
"Type"                     : "H"}

new_data = pd.DataFrame([new_data])
new_data = pd.get_dummies(new_data, columns=["Type"], dtype=int)
new_data = new_data.reindex(columns=feature_columns, fill_value=0)
one_hot = new_data[["Type_H", "Type_L", "Type_M"]]
scale = new_data.iloc[:, :5]
scaled_data = (scale-mean)/std

x = pd.concat([scaled_data, one_hot], axis=1)
print("Given data: \n",x)
x_tensor = torch.tensor(x.values,dtype =torch.float32)

x_tensor = x_tensor.to(device)

model.eval()
with torch.no_grad():
    prediction = model(x_tensor)
    probabilities = torch.softmax(prediction, dim=1)
    failure_probability = probabilities[0,1].item()

threshold = 0.61

if failure_probability >= threshold:
    print("Machine Failure: YES")
    print(f"Failure Probability {probabilities[0,1].item()*100:.1f}%")
else:
    print("Machine Failure: NO")
    print(f"Failure Probability {probabilities[0,0].item()*100:.1f}%")


Given data: 
    Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0             -0.89385                -0.409143               -0.549236   

   Torque [Nm]  Tool wear [min]  Type_H  Type_L  Type_M  
0    -0.279383        -0.784578       1       0       0  
Machine Failure: NO
Failure Probability 99.7%
